In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. Load the upgraded Saturday Night Predictor Table
df_all = spark.read.table("gold_saturday_predictor_features").toPandas()

# 2. Fix the Data Type Trap
df_all['Year'] = df_all['Year'].astype(int)

# 3. Re-create the Pace Rank
df_all['Race_Pace_Rank'] = df_all.groupby(['Year', 'Track'])['Pace_Delta_To_Leader'].rank(method='dense')

# 4. Create the Target Variable (Podium Contender)
df_all['Is_Podium_Contender'] = np.where(
    (df_all['Simulated_Grid_Position'] <= 5) & (df_all['Race_Pace_Rank'] <= 5), 1, 0
)

# 5. TIME-SERIES SPLIT: Train the AI strictly on the Past
df_train = df_all[df_all['Year'] < 2026]

# 6. Extract the Future (Australia 2026) for the Blind Test
df_test = df_all[(df_all['Year'] == 2026) & (df_all['Track'] == 'china')].copy()

# 7. Define the Features (Notice we DON'T feed the AI the 'Source Session' string, just the pure math)
features = ['Simulated_Grid_Position', 'Qualifying_Delta_To_Pole', 'Pace_Delta_To_Leader']
X_train = df_train[features]
y_train = df_train['Is_Podium_Contender']

print("🧠 Booting up the Engine (Now with Sprint-Weekend Fallback Logic)...")
model = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)
print("✅ AI Trained!\n")

# 8. THE FINAL TEST
if df_test.empty:
    print("⚠️ No data for China. Change 'china' to 'shanghai' on Line 26!")
else:
    X_live = df_test[features]
    
    # Get the confidence percentages
    probabilities = model.predict_proba(X_live)[:, 1] 

    # Format and rank the results
    df_test['Win_Probability'] = (probabilities * 100).round(1)
    predictions_aus = df_test.sort_values(by='Win_Probability', ascending=False)

    print("🏁 2026 CHINESE GRAND PRIX - FINAL PREDICTIONS 🏁")
    print("=================================================================")
    # Notice we added 'Pace_Source_Session' to the display so we can see where the data came from!
    display(predictions_aus[['Team', 'DriverNumber', 'Simulated_Grid_Position', 'Pace_Source_Session', 'Pace_Delta_To_Leader', 'Win_Probability']].head(10))

StatementMeta(, 6d9dc7ae-66ca-438a-9996-78cde793b649, 5, Finished, Available, Finished, False)

🧠 Booting up the Engine (Now with Sprint-Weekend Fallback Logic)...


✅ AI Trained!

🏁 2026 CHINESE GRAND PRIX - FINAL PREDICTIONS 🏁


SynapseWidget(Synapse.DataFrame, 2335fd10-e58c-461d-ab07-e95efa7f3426)